# 宜蘭高麗菜價格預測：雨量與時間特徵的模型比較

## 專案目標
- 利用雨量、時間與歷史價格特徵，預測高麗菜價格是否上漲
- 比較 Logistic Regression、Random Forest、LightGBM 三種模型表現
- 透過特徵重要性分析，了解哪些因素最影響價格變動

## 任務定義
- 任務類型：二元分類
- 預測目標：隔日價格是否高於今日價格
- 評估指標：Accuracy、Precision、Recall、F1-score

## 專案提醒
- 這份資料具有時間順序，切分資料時不要隨機打亂
- 若之後加入更多天氣欄位（溫度、濕度、日照），可直接擴充到同一流程中


## 1. 載入套件
先集中管理本專案會用到的套件。

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# 如果還沒安裝 lightgbm，可先註解掉下一行
from lightgbm import LGBMClassifier


## 2. 讀取資料
請把這裡改成你自己的資料路徑。

In [ ]:
# TODO: 修改成你的實際檔案路徑
df = pd.read_csv('data/processed/cabbage_yilan_weather.csv')

df.head()

## 3. 基本資料檢查
先確認欄位、資料型態、缺失值與資料筆數。

In [ ]:
print('資料筆數與欄數:', df.shape)
display(df.head())
display(df.info())
display(df.isnull().sum())
display(df.describe(include='all'))

### 小結
- 這裡記錄你看到的資料狀況
- 例如：哪些欄位有缺值、日期是否需要轉型、價格分布是否偏斜

## 4. 欄位整理與日期處理
確認日期欄位格式，並依時間排序。

In [ ]:
# TODO: 修改 date 欄位名稱
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

df[['date']].head()

## 5. Exploratory Data Analysis (EDA)
先看價格變化、雨量分布，以及兩者關係。

In [ ]:
plt.figure(figsize=(12, 4))
# TODO: 修改 price 欄位名稱
plt.plot(df['date'], df['price'])
plt.title('高麗菜價格時間走勢')
plt.xlabel('Date')
plt.ylabel('Price')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
# TODO: 修改 rainfall 欄位名稱
plt.hist(df['rainfall'].dropna(), bins=30)
plt.title('雨量分布')
plt.xlabel('Rainfall')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(df['rainfall'], df['price'], alpha=0.5)
plt.title('雨量與價格關係')
plt.xlabel('Rainfall')
plt.ylabel('Price')
plt.tight_layout()
plt.show()

### EDA 觀察
- 這裡補上你對圖表的觀察
- 例如：是否有季節性、極端雨量、價格劇烈波動區段

## 6. Feature Engineering
建立時間特徵、歷史價格特徵與雨量聚合特徵。

In [ ]:
# 時間特徵
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['dayofweek'] = df['date'].dt.dayofweek

# 歷史價格特徵
df['price_lag_1'] = df['price'].shift(1)
df['price_lag_7'] = df['price'].shift(7)
df['price_rolling_mean_7'] = df['price'].shift(1).rolling(7).mean()

# 雨量特徵
df['rainfall_lag_1'] = df['rainfall'].shift(1)
df['rainfall_rolling_sum_7'] = df['rainfall'].shift(1).rolling(7).sum()
df['rainfall_rolling_mean_7'] = df['rainfall'].shift(1).rolling(7).mean()

display(df.head(10))

## 7. 建立目標欄位
這裡先用分類版本：預測隔日價格是否上漲。

In [ ]:
df['target'] = (df['price'].shift(-1) > df['price']).astype(int)

display(df[['date', 'price', 'target']].head(10))
display(df['target'].value_counts())

## 8. 選擇特徵與清理缺失值

In [ ]:
feature_cols = [
    'rainfall',
    'month',
    'dayofweek',
    'price_lag_1',
    'price_lag_7',
    'price_rolling_mean_7',
    'rainfall_lag_1',
    'rainfall_rolling_sum_7',
    'rainfall_rolling_mean_7'
]

model_df = df[feature_cols + ['target', 'date']].dropna().copy()

X = model_df[feature_cols]
y = model_df['target']

print('建模資料 shape:', model_df.shape)
display(model_df.head())

## 9. Train / Test Split
因為是時間資料，使用前段做訓練、後段做測試。

In [ ]:
split_index = int(len(model_df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print('X_train:', X_train.shape)
print('X_test :', X_test.shape)
print('y_train:', y_train.shape)
print('y_test :', y_test.shape)
print('訓練集期間:', model_df.iloc[:split_index]['date'].min(), '~', model_df.iloc[:split_index]['date'].max())
print('測試集期間:', model_df.iloc[split_index:]['date'].min(), '~', model_df.iloc[split_index:]['date'].max())

## 10. 定義評估函式
統一整理每個模型的指標，避免重複貼很多同樣的程式碼。

In [ ]:
def evaluate_model(model_name, y_true, y_pred):
    return {
        'Model': model_name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0)
    }


## 11. Logistic Regression（Baseline）

In [ ]:
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)

print('Logistic Regression')
print(classification_report(y_test, y_pred_lr, zero_division=0))
print(confusion_matrix(y_test, y_pred_lr))

## 12. Random Forest

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    random_state=42
)

rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print('Random Forest')
print(classification_report(y_test, y_pred_rf, zero_division=0))
print(confusion_matrix(y_test, y_pred_rf))

## 13. LightGBM

In [ ]:
lgbm_model = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    random_state=42
)

lgbm_model.fit(X_train, y_train)
y_pred_lgbm = lgbm_model.predict(X_test)

print('LightGBM')
print(classification_report(y_test, y_pred_lgbm, zero_division=0))
print(confusion_matrix(y_test, y_pred_lgbm))

## 14. 模型比較表

In [ ]:
results = pd.DataFrame([
    evaluate_model('Logistic Regression', y_test, y_pred_lr),
    evaluate_model('Random Forest', y_test, y_pred_rf),
    evaluate_model('LightGBM', y_test, y_pred_lgbm)
])

results = results.sort_values('F1', ascending=False).reset_index(drop=True)
display(results)

### 比較觀察
- 哪個模型最好？
- 是 Precision 比較高，還是 Recall 比較高？
- 是否代表這份資料存在非線性關係？

## 15. Feature Importance

In [ ]:
rf_importance = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(8, 4))
rf_importance.plot(kind='bar')
plt.title('Random Forest Feature Importance')
plt.tight_layout()
plt.show()

display(rf_importance)

In [ ]:
lgbm_importance = pd.Series(lgbm_model.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(8, 4))
lgbm_importance.plot(kind='bar')
plt.title('LightGBM Feature Importance')
plt.tight_layout()
plt.show()

display(lgbm_importance)

## 16. 結論

### 模型結果
- 在這裡整理哪個模型表現最好
- 嘗試說明為什麼該模型表現較佳

### 特徵觀察
- 哪些欄位最重要？
- 雨量是否真的有幫助？
- 歷史價格特徵是否比天氣更重要？

### 後續改進方向
- 加入更多氣象特徵（溫度、濕度、日照）
- 嘗試回歸版本（預測實際價格）
- 使用時間序列交叉驗證
- 加入 permutation importance / SHAP 進一步解釋模型
